# MAT204 – Probability and Statistics for Engineers
## Chapter 5 – Continuous Random Variables

**Dr. Haydar Kılıç | OSTİMTECH Faculty of Engineering**

---

| Part | Topics |
|------|--------|
| **Part 1** | PDF, CDF, Expected Value & Variance, Uniform Distribution |
| **Part 2** | Normal Distribution, z-Transform, Normal Approximation to Binomial, Distribution of a Function |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon
from scipy import stats, integrate
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})
np.random.seed(42)
print("Libraries loaded successfully ✓")

---
## Part 1 – PDF, CDF, and Uniform Distribution

### 1.1 What Is a Continuous Random Variable?

> **Definition:** If $X$ can take any real value in $(-\\infty, \\infty)$ and there exists a non-negative function $f$ such that:
> $$P(X \\in B) = \\int_B f(x)\\,dx$$
> then $f$ is called the **Probability Density Function (PDF)**.

**Discrete ↔ Continuous comparison:**

| Property | Discrete | Continuous |
|----------|----------|------------|
| Values | Countable | Uncountable |
| Definition | PMF: $p(x) = P(X=x)$ | PDF: $f(x)$, $P(X \\in B) = \\int_B f(x)dx$ |
| Single point | $P(X=a)$ can be $>0$ | $P(X=a) = 0$ always |
| Total | $\\sum p(x) = 1$ | $\\int_{-\\infty}^{\\infty} f(x)dx = 1$ |

In [ ]:
# Visualizing the two required conditions of a PDF
# Condition 1: f(x) >= 0
# Condition 2: integral = 1

# Example: f(x) = 2x, 0 <= x <= 1
x = np.linspace(0, 1, 300)
f_x = 2 * x

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: PDF plot + shading
ax = axes[0]
ax.plot(x, f_x, 'steelblue', linewidth=2.5)
ax.fill_between(x, f_x, alpha=0.25, color='steelblue')
ax.set_xlim(-0.1, 1.2); ax.set_ylim(-0.05, 2.2)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('PDF: f(x) = 2x  (0 <= x <= 1)')
ax.text(0.5, 1.0, 'Area = 1', ha='center', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Condition 2 verification
integral, _ = integrate.quad(lambda x: 2*x, 0, 1)
print("PDF validity:")
print("  1) f(x) >= 0?  Yes, 2x >= 0 for x >= 0 ✓")
print(f"  2) Integral = 1? {integral:.6f}  ✓")

# Right: finding the constant c
# f(x) = c(8x - 4x^3), 0 <= x <= 1
ax = axes[1]
x_fine = np.linspace(0, 1, 300)
c = 1/3
f_c = c * (8*x_fine - 4*x_fine**3)
ax.plot(x_fine, f_c, 'tomato', linewidth=2.5)
ax.fill_between(x_fine, f_c, alpha=0.25, color='tomato')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('PDF: f(x) = (1/3)(8x - 4x^3)  ->  c = 1/3')
ax.text(0.4, 0.9, 'c = 1/3', ha='center', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.axhline(0, color='black', linewidth=0.8)

integral2, _ = integrate.quad(lambda x: c*(8*x - 4*x**3), 0, 1)
print(f"\nIntegral for c(8x-4x^3): c*{integral2/c:.4f} = {integral2:.6f} ✓")
plt.tight_layout(); plt.show()

### 1.2 Relationship Between CDF and PDF

**Cumulative Distribution Function (CDF):**
$$F(a) = P(X \\le a) = \\int_{-\\infty}^{a} f(x)\\,dx$$

**PDF to CDF:** $F(a) = \\int_{-\\infty}^{a} f(x)dx$

**CDF to PDF:** $f(x) = \\dfrac{d}{dx}F(x)$

**Important:** For a continuous variable, the probability at a single point is **zero:**
$$P(X = a) = \\int_a^a f(x)dx = 0 \\implies P(a \\le X \\le b) = P(a < X < b)$$

In [ ]:
# Show PDF and CDF together for the f(x) = 2x example
x_range = np.linspace(-0.2, 1.3, 500)
x_inner = np.linspace(0, 1, 300)

# PDF
def pdf(x):
    return np.where((x >= 0) & (x <= 1), 2*x, 0.0)

# CDF: F(x) = x^2 for 0<=x<=1
def cdf(x):
    return np.where(x < 0, 0.0, np.where(x > 1, 1.0, x**2))

# Interval probability: P(0.3 <= X <= 0.7)
a, b = 0.3, 0.7
prob_ab, _ = integrate.quad(lambda x: 2*x, a, b)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# PDF + shaded interval
ax = axes[0]
ax.plot(x_range, pdf(x_range), 'steelblue', linewidth=2.5, label='f(x) = 2x')
x_shade = np.linspace(a, b, 200)
ax.fill_between(x_shade, pdf(x_shade), alpha=0.5, color='orange', label=f'P({a}<=X<={b})')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('PDF: f(x) = 2x')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); ax.set_xlim(-0.1, 1.2)

# CDF
ax = axes[1]
ax.plot(x_range, cdf(x_range), 'tomato', linewidth=2.5, label='F(x) = x^2')
ax.axhline(cdf(b), color='gray', linestyle=':', linewidth=1)
ax.axhline(cdf(a), color='gray', linestyle=':', linewidth=1)
ax.annotate('', xy=(1.15, cdf(b)), xytext=(1.15, cdf(a)),
            arrowprops=dict(arrowstyle='<->', color='orange', lw=2))
ax.text(1.17, (cdf(a)+cdf(b))/2, f'{prob_ab:.4f}', va='center', color='darkorange', fontsize=10)
ax.set_title('CDF: F(x) = x^2')
ax.set_xlabel('x'); ax.set_ylabel('F(x)')
ax.legend(); ax.set_xlim(-0.1, 1.3)

plt.tight_layout(); plt.show()

print(f"P({a} <= X <= {b}) = F({b}) - F({a}) = {b**2:.4f} - {a**2:.4f} = {prob_ab:.4f}")
print(f"Verification: integral = {prob_ab:.4f}  ✓")

### 1.3 Interpretation of the PDF

For a small $h > 0$:
$$P\\left(x - \\frac{h}{2} \\le X \\le x + \\frac{h}{2}\\right) \\approx h \\cdot f(x)$$

The larger $f(x)$, the higher the probability that $X$ is **close** to $x$.

You can see this illustrated in the next cell.

In [ ]:
# PDF interpretation: small interval h approximation
fig, ax = plt.subplots(figsize=(10, 4.5))

x_range = np.linspace(0, 1, 500)
ax.plot(x_range, 2*x_range, 'steelblue', linewidth=2.5, label='f(x) = 2x')
ax.axhline(0, color='black', linewidth=0.8)

# Show h*f(x) at different points
colors = ['orange', 'green', 'red']
x_points = [0.2, 0.5, 0.8]
h = 0.05
for xi, color in zip(x_points, colors):
    p_approx = h * 2 * xi
    p_exact, _ = integrate.quad(lambda x: 2*x, xi-h/2, xi+h/2)
    ax.fill_between([xi-h/2, xi+h/2], [2*(xi-h/2), 2*(xi+h/2)],
                   alpha=0.6, color=color)
    ax.axvline(xi, color=color, linestyle='--', linewidth=1, alpha=0.7)
    ax.text(xi, 2*xi + 0.08, f"x={xi}\nh*f(x)={p_approx:.4f}\nexact={p_exact:.4f}",
            ha='center', fontsize=8, color=color,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_title('PDF Interpretation: P(x-h/2 <= X <= x+h/2) ≈ h·f(x)  (h=0.05)')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.legend(); plt.tight_layout(); plt.show()

### 1.4 Expected Value and Variance of a Continuous Random Variable

| | Discrete | Continuous |
|--|----------|------------|
| $E[X]$ | $\\sum_x x\\, p(x)$ | $\\int_{-\\infty}^{\\infty} x\\, f(x)\\,dx$ |
| $E[g(X)]$ | $\\sum_x g(x)\\,p(x)$ | $\\int_{-\\infty}^{\\infty} g(x)\\,f(x)\\,dx$ |
| $\\text{Var}(X)$ | $E[X^2] - (E[X])^2$ | $E[X^2] - (E[X])^2$ |

**Properties (same as discrete):**
$$E[aX+b] = aE[X]+b \\qquad \\text{Var}(aX+b) = a^2\\text{Var}(X) \\qquad E[X+Y]=E[X]+E[Y]$$

In [ ]:
# Example: f(x) = 2x, 0<=x<=1
from scipy.integrate import quad

f = lambda x: 2*x

E_X,  _ = quad(lambda x: x * f(x),    0, 1)
E_X2, _ = quad(lambda x: x**2 * f(x), 0, 1)
Var_X   = E_X2 - E_X**2
SD_X    = np.sqrt(Var_X)

print("f(x) = 2x  (0 <= x <= 1)")
print("-"*40)
print(f"  E[X]   = integral(x*2x, 0,1) = {E_X:.4f}  (analytic: 2/3 = {2/3:.4f})")
print(f"  E[X^2] = integral(x^2*2x,0,1) = {E_X2:.4f}  (analytic: 1/2 = {1/2:.4f})")
print(f"  Var(X) = E[X^2] - E[X]^2 = {E_X2:.4f} - {E_X**2:.4f} = {Var_X:.4f}  (analytic: 1/18 = {1/18:.4f})")
print(f"  SD(X)  = sqrt(Var(X)) = {SD_X:.4f}")

# Linearity verification
a, b = 3.0, 2.0
E_aXb_direct, _ = quad(lambda x: (a*x + b) * f(x), 0, 1)
E_aXb_rule      = a * E_X + b
print(f"\nLinearity rule E[3X+2]:")
print(f"  Direct integral : {E_aXb_direct:.4f}")
print(f"  3*E[X]+2        : {E_aXb_rule:.4f}")
print(f"  Equal? {chr(10003) if np.isclose(E_aXb_direct, E_aXb_rule) else chr(10007)}")

### 1.5 Uniform Distribution

> **Definition:** $X \\sim \\text{Unif}(\\alpha, \\beta)$

$$f(x) = \\frac{1}{\\beta-\\alpha} \\cdot \\mathbf{1}_{(\\alpha,\\beta)}(x), \\qquad F(x) = \\frac{x-\\alpha}{\\beta-\\alpha} \\;\\text{ for }\\; \\alpha < x < \\beta$$

$$E[X] = \\frac{\\alpha+\\beta}{2}, \\qquad \\text{Var}(X) = \\frac{(\\beta-\\alpha)^2}{12}$$

**Derivation:** $c = 1/(\\beta-\\alpha)$ because $\\int_\\alpha^\\beta c\\,dx = c(\\beta-\\alpha) = 1$

In [ ]:
def unif_pdf(x, a, b):
    return np.where((x > a) & (x < b), 1.0/(b-a), 0.0)

def unif_cdf(x, a, b):
    return np.where(x <= a, 0.0, np.where(x >= b, 1.0, (x-a)/(b-a)))

# Multiple interval combinations
configs = [(0, 1), (2, 5), (-2, 3)]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, (a, b) in enumerate(configs):
    mu  = (a + b) / 2
    var = (b - a)**2 / 12
    sd  = np.sqrt(var)
    x   = np.linspace(a - 0.5, b + 0.5, 500)

    ax = axes[0, i]
    ax.plot(x, unif_pdf(x, a, b), 'steelblue', linewidth=2.5)
    ax.fill_between(x, unif_pdf(x, a, b), alpha=0.25, color='steelblue')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Unif({a},{b})  PDF\nf(x)=1/{b-a}')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')

    ax = axes[1, i]
    ax.plot(x, unif_cdf(x, a, b), 'tomato', linewidth=2.5)
    ax.axhline(1, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(f'Unif({a},{b})  CDF\nmu={mu:.2f}, sigma^2={var:.4f}')
    ax.set_xlabel('x'); ax.set_ylabel('F(x)')

plt.suptitle('Uniform Distribution – PDF and CDF', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Analytic verification
print("Unif(alpha, beta) analytic values:")
print(f"  E[X] = (alpha+beta)/2")
print(f"  Var(X) = (beta-alpha)^2/12")

for a, b in configs:
    mu_analytic  = (a+b)/2
    var_analytic = (b-a)**2/12
    mu_num, _    = quad(lambda x: x/(b-a), a, b)
    ex2, _       = quad(lambda x: x**2/(b-a), a, b)
    var_num      = ex2 - mu_num**2
    print(f"  Unif({a},{b}): E[X]={mu_analytic:.4f} ({mu_num:.4f}), Var={var_analytic:.4f} ({var_num:.4f}) ✓")

In [ ]:
# Bus example: Unif(0,60) – probability of waiting more than 7 minutes
alpha, beta = 0, 60
n_buses = 6  # every 10 minutes: 0,10,20,30,40,50

fig, ax = plt.subplots(figsize=(13, 4))
x = np.linspace(-2, 62, 500)
ax.plot(x, unif_pdf(x, alpha, beta), 'steelblue', linewidth=2.5)
ax.fill_between(x, unif_pdf(x, alpha, beta), alpha=0.1, color='steelblue')

# Intervals where waiting > 7 min: bus arrives at t=0,10,...,50
# Waiting > 7 min: t in (0,3), (10,13), (20,23), (30,33), (40,43), (50,53)
wait_intervals = [(0,3), (10,13), (20,23), (30,33), (40,43), (50,53)]
total_length = 0
for (t1, t2) in wait_intervals:
    x_s = np.linspace(t1, t2, 100)
    ax.fill_between(x_s, unif_pdf(x_s, alpha, beta),
                   alpha=0.7, color='tomato')
    ax.axvline(t1, color='gray', linestyle=':', linewidth=0.8)
    ax.axvline(t2, color='red', linestyle='--', linewidth=0.8)
    total_length += (t2 - t1)

for t_bus in range(0, 61, 10):
    ax.axvline(t_bus, color='green', linestyle='-', linewidth=1.5,
               alpha=0.7, label='Bus' if t_bus == 0 else '_')

red_patch   = mpatches.Patch(color='tomato', alpha=0.7, label='>7 min wait')
green_patch = mpatches.Patch(color='green',  alpha=0.7, label='Bus arrival')
ax.legend(handles=[red_patch, green_patch])
ax.set_xlabel('Arrival time (minutes)')
ax.set_ylabel('f(x) = 1/60')
ax.set_title('Bus Problem: X ~ Unif(0,60)  – Probability of waiting more than 7 minutes')
ax.set_xlim(-2, 62)
plt.tight_layout(); plt.show()

P_7min = total_length / 60
print(f"Total length of red intervals: {total_length} minutes")
print(f"P(wait > 7 min) = {total_length}/60 = {P_7min:.4f}")

---
## Part 2 – Normal Distribution

### 2.1 Definition of the Normal Distribution

> **Definition:** $X \\sim N(\\mu, \\sigma^2)$:
> $$f(x) = \\frac{1}{\\sqrt{2\\pi\\sigma^2}}\\, e^{-\\frac{(x-\\mu)^2}{2\\sigma^2}}, \\quad x \\in \\mathbb{R}$$

**Properties:**
- Unimodal, symmetric (bell-shaped)
- $E[X] = \\mu$, $\\text{Var}(X) = \\sigma^2$
- Symmetry: $f(\\mu - x) = f(\\mu + x)$
- **68-95-99.7 Rule:** $P(|X-\\mu| \\le k\\sigma) \\approx$ 68%, 95%, 99.7% for $k=1,2,3$

In [ ]:
# Normal distribution – different parameter combinations
configs = [
    (0,  0.2, 'N(0, 0.2)',  'steelblue'),
    (0,  1.0, 'N(0, 1.0)',  'tomato'),
    (0,  5.0, 'N(0, 5.0)',  'seagreen'),
    (-2, 0.5, 'N(-2, 0.5)', 'darkorange'),
]

x = np.linspace(-6, 4, 600)
fig, ax = plt.subplots(figsize=(12, 5))

for mu, sig2, lbl, color in configs:
    pdf_vals = stats.norm.pdf(x, mu, np.sqrt(sig2))
    ax.plot(x, pdf_vals, color=color, linewidth=2, label=lbl)

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Normal Distribution PDF – Different Parameters')
ax.legend(); plt.tight_layout(); plt.show()

print("Standard normal N(0,1):")
mu, sig = 0, 1
for k in [1, 2, 3]:
    p = stats.norm.cdf(mu+k*sig) - stats.norm.cdf(mu-k*sig)
    print(f"  P(|X-mu| <= {k}*sigma) = {p*100:.2f}%")

### 2.2 The 68-95-99.7 Rule (Empirical Rule)

$$P(\\mu - \\sigma < X < \\mu + \\sigma) \\approx 68\\%$$
$$P(\\mu - 2\\sigma < X < \\mu + 2\\sigma) \\approx 95\\%$$
$$P(\\mu - 3\\sigma < X < \\mu + 3\\sigma) \\approx 99.7\\%$$

In [ ]:
mu, sigma = 0, 1
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 600)
pdf_vals = stats.norm.pdf(x, mu, sigma)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, pdf_vals, 'black', linewidth=2)

regions = [
    (1, 'steelblue',  '68%  (±1σ)'),
    (2, 'lightblue',  '95%  (±2σ)'),
    (3, 'lightyellow','99.7% (±3σ)'),
]
for k, color, lbl in reversed(regions):
    x_fill = np.linspace(mu-k*sigma, mu+k*sigma, 400)
    ax.fill_between(x_fill, stats.norm.pdf(x_fill, mu, sigma),
                   alpha=0.7, color=color, label=lbl)

for k in [1, 2, 3]:
    for sign in [-1, 1]:
        xp = mu + sign*k*sigma
        ax.axvline(xp, color='gray', linestyle='--', linewidth=0.8)
        _lbl = "mu" if k == 0 else ("" if sign > 0 else "-") + f"{k}s"
        ax.text(xp, -0.018, _lbl, ha='center', fontsize=8, fontweight='bold')

ax.axvline(mu, color='black', linestyle='-', linewidth=1.2)
ax.text(mu, -0.018, 'mu', ha='center', fontsize=9, fontweight='bold')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('68-95-99.7 Rule – N(0,1)')
ax.legend(loc='upper right')
ax.set_ylim(-0.03, 0.45)
plt.tight_layout(); plt.show()

### 2.3 Standard Normal and z-Transform

**Standard normal:** $Z \\sim N(0,1)$, PDF $\\phi(x)$, CDF $\\Phi(x)$

**z-Transform:** $X \\sim N(\\mu, \\sigma^2) \\Rightarrow Z = \\dfrac{X-\\mu}{\\sigma} \\sim N(0,1)$

**Probability computation:**
$$P(a < X < b) = \\Phi\\left(\\frac{b-\\mu}{\\sigma}\\right) - \\Phi\\left(\\frac{a-\\mu}{\\sigma}\\right)$$

**Symmetry properties:**
$$\\Phi(-z) = 1 - \\Phi(z) \\qquad P(|Z| > z) = 2(1 - \\Phi(z))$$

In [ ]:
# z-transform visualization
mu, sigma = 19, 4  # sigma^2 = 16

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# X ~ N(19, 16)
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
ax = axes[0]
ax.plot(x, stats.norm.pdf(x, mu, sigma), 'steelblue', linewidth=2.5)
ax.fill_between(x, stats.norm.pdf(x, mu, sigma), alpha=0.2, color='steelblue')
ax.set_title(f'X ~ N(mu={mu}, sigma^2={sigma**2})')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
for k in range(-3, 4):
    ax.axvline(mu + k*sigma, color='gray', linestyle=':', linewidth=0.8)
    ax.text(mu + k*sigma, -0.008,
            f'{mu+k*sigma}', ha='center', fontsize=7)

# Z ~ N(0,1)
z = np.linspace(-4, 4, 500)
ax = axes[1]
ax.plot(z, stats.norm.pdf(z), 'tomato', linewidth=2.5)
ax.fill_between(z, stats.norm.pdf(z), alpha=0.2, color='tomato')
ax.set_title('Z ~ N(0,1)  (z = (X-mu)/sigma)')
ax.set_xlabel('z'); ax.set_ylabel('phi(z)')
for k in range(-3, 4):
    ax.axvline(k, color='gray', linestyle=':', linewidth=0.8)
    ax.text(k, -0.03, str(k), ha='center', fontsize=8)

plt.suptitle('z-Transform: X = sigma*Z + mu', fontsize=13)
plt.tight_layout(); plt.show()

# Phi table – z lookups
print("Standard Normal Phi(z) table examples:")
z_vals = [-2.0, -1.82, -1.5, 0.0, 1.0, 1.64, 1.96, 2.0]
print(f"  {'z':>6} | {'Phi(z)':>8} | {'1-Phi(z)':>10}")
print("  " + "-"*30)
for z_val in z_vals:
    print(f"  {z_val:>6.2f} | {stats.norm.cdf(z_val):>8.4f} | {1-stats.norm.cdf(z_val):>10.4f}")

### 2.4 Heinz Ketchup Example

$X \\sim N(\\mu=36, \\sigma=0.11)$ — amount of ketchup in a bottle (oz)

**Question:** $P(X < 35.8)$ and $P(\\text{quality control failure}) = P(X < 35.8 \\text{ or } X > 36.2)$

In [ ]:
mu, sigma = 36, 0.11
a_lower, a_upper = 35.8, 36.2

# z-score computation
z_lower = (a_lower - mu) / sigma
z_upper = (a_upper - mu) / sigma

P_lower = stats.norm.cdf(z_lower)
P_upper = 1 - stats.norm.cdf(z_upper)
P_fail  = P_lower + P_upper
P_pass  = 1 - P_fail

print("Heinz Ketchup Example:")
print(f"  X ~ N(mu={mu}, sigma={sigma})")
print(f"  z_lower = ({a_lower} - {mu}) / {sigma} = {z_lower:.4f}")
print(f"  z_upper = ({a_upper} - {mu}) / {sigma} = {z_upper:.4f}")
print(f"  P(X < {a_lower}) = Phi({z_lower:.2f}) = {P_lower:.4f}")
print(f"  P(X > {a_upper}) = 1 - Phi({z_upper:.2f}) = {P_upper:.4f}")
print(f"  P(quality control FAIL) = {P_fail:.4f}")
print(f"  P(quality control PASS) = {P_pass:.4f}")

# Visualization
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
pdf_vals = stats.norm.pdf(x, mu, sigma)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, title, fill_color, region in zip(
    axes,
    ['P(X < 35.8)  (lower limit)', 'P(X < 35.8 or X > 36.2)  (quality control)'],
    ['tomato', 'tomato'],
    ['left', 'both']
):
    ax.plot(x, pdf_vals, 'steelblue', linewidth=2)
    if region == 'left':
        x_fill = np.linspace(mu-4*sigma, a_lower, 200)
        ax.fill_between(x_fill, stats.norm.pdf(x_fill, mu, sigma),
                       alpha=0.6, color='tomato')
    else:
        x_f1 = np.linspace(mu-4*sigma, a_lower, 200)
        x_f2 = np.linspace(a_upper, mu+4*sigma, 200)
        ax.fill_between(x_f1, stats.norm.pdf(x_f1, mu, sigma),
                       alpha=0.6, color='tomato')
        ax.fill_between(x_f2, stats.norm.pdf(x_f2, mu, sigma),
                       alpha=0.6, color='tomato')
        x_ok = np.linspace(a_lower, a_upper, 200)
        ax.fill_between(x_ok, stats.norm.pdf(x_ok, mu, sigma),
                       alpha=0.3, color='seagreen', label='Valid')
    ax.axvline(a_lower, color='red', linestyle='--', linewidth=1.5)
    ax.axvline(a_upper, color='red', linestyle='--', linewidth=1.5)
    ax.axvline(mu, color='black', linestyle='-', linewidth=1.2)
    ax.set_title(title)
    ax.set_xlabel('Ketchup amount (oz)')

plt.suptitle('N(mu=36, sigma=0.11) – Quality Control', fontsize=12)
plt.tight_layout(); plt.show()

### 2.5 Normal Approximation to the Binomial

For $X \\sim \\text{Bin}(n,p)$ with $np(1-p) \\ge 10$:

$$\\text{Bin}(n,p) \\approx N(\\mu=np,\\; \\sigma^2=np(1-p))$$

**Continuity correction** (better approximation):
$$P(a \\le X \\le b) \\approx P(a-0.5 < Y < b+0.5) = \\Phi\\!\\left(\\frac{b+0.5-np}{\\sqrt{np(1-p)}}\\right) - \\Phi\\!\\left(\\frac{a-0.5-np}{\\sqrt{np(1-p)}}\\right)$$

In [ ]:
# University example: Bin(450, 0.3) -> P(X > 150)
n, p_prob = 450, 0.3
mu_b  = n * p_prob
sig_b = np.sqrt(n * p_prob * (1 - p_prob))

print("University admissions example: X ~ Bin(n=450, p=0.3)")
print(f"  mu = np = {mu_b}, sigma = sqrt(np(1-p)) = {sig_b:.4f}")
print(f"  np(1-p) = {n*p_prob*(1-p_prob):.1f} >= 10  ✓ (Normal approximation applicable)")

# Exact Binomial
P_exact = 1 - stats.binom.cdf(150, n, p_prob)

# Without continuity correction
z_150  = (150 - mu_b) / sig_b
P_ncc  = 1 - stats.norm.cdf(z_150)

# With continuity correction
z_150c = (150 + 0.5 - mu_b) / sig_b
P_cc   = 1 - stats.norm.cdf(z_150c)

print(f"\n  P(X > 150) exact Binomial              = {P_exact:.4f}")
print(f"  P(X > 150) Normal (no correction)      = {P_ncc:.4f}  (z={z_150:.4f})")
print(f"  P(X > 150) Normal (with correction)    = {P_cc:.4f}  (z={z_150c:.4f})")
print(f"  Textbook answer: 0.0559 -- {chr(10003)}")

# Visualization: Binomial + Normal overlay
k_vals    = np.arange(100, 200)
pmf_binom = stats.binom.pmf(k_vals, n, p_prob)
x_cont    = np.linspace(95, 200, 500)
pdf_norm  = stats.norm.pdf(x_cont, mu_b, sig_b)

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(k_vals, pmf_binom, width=0.8, alpha=0.5, color='steelblue',
       label=f'Bin({n},{p_prob})')
ax.plot(x_cont, pdf_norm, 'tomato', linewidth=2.5,
        label=f'N({mu_b},{sig_b**2:.1f})')
ax.axvline(150, color='red', linestyle='--', linewidth=2, label='X=150')
ax.axvline(150.5, color='orange', linestyle='--', linewidth=2,
           label='X=150.5 (with correction)')
ax.fill_betweenx([0, max(pmf_binom)*1.1], 150, 200, alpha=0.1, color='red')
ax.set_xlabel('k'); ax.set_ylabel('Probability')
ax.set_title('Binomial -> Normal Approximation (with Continuity Correction)')
ax.legend(); plt.tight_layout(); plt.show()

### 2.6 Distribution of a Function (Change of Variables)

Let the PDF of $X$ be $f_X(x)$. If $g(x)$ is **monotone** and **differentiable**, then for $Y = g(X)$:

$$f_Y(y) = f_X\\!\\left(g^{-1}(y)\\right) \\cdot \\left|\\frac{d}{dy}g^{-1}(y)\\right|$$

**Example:** $X \\sim \\text{Unif}(0,1)$, $Y = -\\ln(X)$

$g^{-1}(y) = e^{-y}$, $\\frac{d}{dy}e^{-y} = -e^{-y}$

$$f_Y(y) = 1 \\cdot |-e^{-y}| = e^{-y} \\implies Y \\sim \\text{Exp}(1)$$

In [ ]:
# Verification of the theorem via simulation
N = 50000
X_samples = np.random.uniform(0, 1, N)
Y_samples = -np.log(X_samples)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# X ~ Unif(0,1)
ax = axes[0]
ax.hist(X_samples, bins=40, density=True, alpha=0.6,
        color='steelblue', edgecolor='white')
x_r = np.linspace(0, 1, 200)
ax.plot(x_r, np.ones_like(x_r), 'tomato', linewidth=2.5)
ax.set_title('X ~ Unif(0,1)')
ax.set_xlabel('x'); ax.set_ylabel('Density')

# -ln(x) transformation curve
ax = axes[1]
x_r2 = np.linspace(0.01, 1, 300)
ax.plot(x_r2, -np.log(x_r2), 'purple', linewidth=2.5)
ax.set_title('g(x) = -ln(x)  (monotone decreasing)')
ax.set_xlabel('x'); ax.set_ylabel('-ln(x)')
ax.fill_between(x_r2, -np.log(x_r2), alpha=0.15, color='purple')

# Y = -ln(X) ~ Exp(1)
ax = axes[2]
y_r = np.linspace(0, 8, 300)
ax.hist(Y_samples, bins=60, density=True, alpha=0.6,
        color='seagreen', edgecolor='white', label='Simulation')
ax.plot(y_r, np.exp(-y_r), 'tomato', linewidth=2.5, label='Exp(1): f(y)=e^{-y}')
ax.set_title('Y = -ln(X) ~ Exp(1)')
ax.set_xlabel('y'); ax.set_ylabel('Density')
ax.legend()

plt.suptitle('Change of Variables Theorem – Simulation Verification', fontsize=13)
plt.tight_layout(); plt.show()

# Verification: Kolmogorov-Smirnov test
ks_stat, ks_p = stats.kstest(Y_samples, 'expon')
print(f"Kolmogorov-Smirnov Test (Y ~ Exp(1)):")
print(f"  Statistic: {ks_stat:.4f}, p-value: {ks_p:.4f}")
result = chr(10003) if ks_p > 0.05 else chr(10007)
print(f"  p > 0.05? {result}  -- Exp(1) hypothesis cannot be rejected")

In [ ]:
# Y = |X| example – CDF method
# X ~ N(0,1), for Y = |X|: fY(y) = fX(y) + fX(-y)

N = 50000
X_norm = np.random.normal(0, 1, N)
Y_abs  = np.abs(X_norm)

y = np.linspace(0, 4, 400)
fY_theoretic = stats.norm.pdf(y, 0, 1) + stats.norm.pdf(-y, 0, 1)  # = 2*phi(y)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
x_range = np.linspace(-4, 4, 400)
ax.hist(X_norm, bins=80, density=True, alpha=0.5, color='steelblue', label='X ~ N(0,1)')
ax.plot(x_range, stats.norm.pdf(x_range), 'blue', linewidth=2, label='phi(x)')
ax.set_title('X ~ N(0,1)')
ax.set_xlabel('x'); ax.legend()

ax = axes[1]
ax.hist(Y_abs, bins=60, density=True, alpha=0.5,
        color='tomato', edgecolor='white', label='Y=|X| simulation')
ax.plot(y, fY_theoretic, 'darkred', linewidth=2.5,
        label='fY(y)=phi(y)+phi(-y)=2phi(y)')
ax.set_title('Y = |X|  –  derived via the CDF method')
ax.set_xlabel('y'); ax.legend(fontsize=8)

plt.suptitle('Distribution of Y=|X|: example of the CDF method', fontsize=12)
plt.tight_layout(); plt.show()

print("Theorem:")
print("  FY(y) = P(|X| < y) = P(-y < X < y) = FX(y) - FX(-y)")
print("  fY(y) = d/dy[FX(y) - FX(-y)] = fX(y) + fX(-y)")
print("  For X~N(0,1): fY(y) = phi(y) + phi(-y) = 2phi(y), y >= 0")

---
## Chapter 5 Summary – Continuous Distributions

| Distribution | Notation | PDF $f(x)$ | $E[X]$ | $\\text{Var}(X)$ |
|-------------|----------|-----------|--------|------------------|
| Uniform | $\\text{Unif}(\\alpha,\\beta)$ | $1/(\\beta-\\alpha)$ | $(\\alpha+\\beta)/2$ | $(\\beta-\\alpha)^2/12$ |
| Normal | $N(\\mu,\\sigma^2)$ | $\\frac{1}{\\sqrt{2\\pi\\sigma^2}}e^{-(x-\\mu)^2/(2\\sigma^2)}$ | $\\mu$ | $\\sigma^2$ |

In [ ]:
# Comparison plot: Uniform vs Normal
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Uniform – different intervals
for ax, (a, b), color in zip(
    axes[0],
    [(0,1),(0,4),(-2,2)],
    ['steelblue','seagreen','darkorange']
):
    mu_u  = (a+b)/2
    sd_u  = np.sqrt((b-a)**2/12)
    x = np.linspace(a-1, b+1, 400)
    ax.plot(x, stats.uniform.pdf(x, a, b-a), color=color, linewidth=2.5)
    ax.fill_between(x, stats.uniform.pdf(x, a, b-a), alpha=0.25, color=color)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'Unif({a},{b})  mu={mu_u:.2f}, sd={sd_u:.3f}')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')

# Normal – different parameters
for ax, (mu, sig), color in zip(
    axes[1],
    [(0,1),(2,0.5),(-1,2)],
    ['tomato','mediumpurple','cadetblue']
):
    x = np.linspace(mu-4*sig, mu+4*sig, 400)
    ax.plot(x, stats.norm.pdf(x, mu, sig), color=color, linewidth=2.5)
    ax.fill_between(x, stats.norm.pdf(x, mu, sig), alpha=0.25, color=color)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(mu, color='black', linestyle='--', linewidth=1, label='mu')
    ax.set_title(f'N(mu={mu}, sigma={sig})  sigma^2={sig**2}')
    ax.set_xlabel('x'); ax.set_ylabel('f(x)')
    ax.legend(fontsize=8)

plt.suptitle('Chapter 5 – Continuous Distributions Overview', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("Chapter 5 complete! ✓")